# Klasifikasi Tingkat Kematangan Pisang

Model: Ultralytics YOLOv8s (Image Classification)  
Dataset: Kaggle Banana Classification  

**Versi Ter-optimasi** — teknik anti-overfitting:
- Augmentasi offline beragam (bukan duplikat identik)
- AdamW + Weight Decay (L2 Regularization)
- Label Smoothing & Dropout
- Cosine LR Scheduler dengan Warmup
- Early Stopping adaptif (patience=20)
- Mixed Precision Training (AMP)


## 1. Instalasi Library


In [ ]:
!pip install ultralytics albumentations -q


## 2. Import Library & Konfigurasi Awal


In [ ]:
import os
import random
import shutil
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import torch

# Seed untuk reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ORIGINAL_DATASET = "/kaggle/input/klasifikasi-pisang/klasifikasi_pisang"
BALANCED_DATASET = "/kaggle/working/balanced_dataset"

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


## 3. Persiapan Dataset (Balanced + Augmentasi Offline)

Kelas minoritas diperbanyak dengan **augmentasi offline** (bukan duplikat identik) agar model tidak menghafal gambar yang sama.


In [ ]:
import albumentations as A
import cv2

# Pipeline augmentasi — kompatibel dengan albumentations terbaru
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-20, 20), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.4),
    A.GaussNoise(p=0.2),
    A.MotionBlur(blur_limit=3, p=0.1),
])

def augment_and_copy(src_img_path, dst_dir, idx):
    """Augmentasi satu gambar dan simpan dengan nama baru."""
    img = cv2.imread(src_img_path)
    if img is None:
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    augmented = aug_pipeline(image=img)["image"]
    out_path = os.path.join(dst_dir, f"aug_{idx}_{os.path.basename(src_img_path)}")
    cv2.imwrite(out_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

if not os.path.exists(BALANCED_DATASET):
    print("Membangun balanced dataset dengan augmentasi offline...")
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(ORIGINAL_DATASET, split)
        if not os.path.isdir(split_dir):
            continue
        new_split_dir = os.path.join(BALANCED_DATASET, split)

        # Hitung jumlah gambar tiap kelas
        class_counts = {}
        for category in os.listdir(split_dir):
            cat_path = os.path.join(split_dir, category)
            if os.path.isdir(cat_path):
                class_counts[category] = len(glob.glob(os.path.join(cat_path, '*')))

        target = max(class_counts.values()) if split == 'train' else None

        for category in os.listdir(split_dir):
            cat_dir = os.path.join(split_dir, category)
            if not os.path.isdir(cat_dir):
                continue
            new_cat_dir = os.path.join(new_split_dir, category)
            os.makedirs(new_cat_dir, exist_ok=True)
            images = glob.glob(os.path.join(cat_dir, '*.*'))

            # Salin gambar asli
            for img in images:
                shutil.copy(img, new_cat_dir)

            # Augmentasi untuk kelas minoritas (hanya split train)
            if split == 'train' and target:
                current_count = len(os.listdir(new_cat_dir))
                aug_idx = 0
                while current_count < target:
                    src = random.choice(images)
                    augment_and_copy(src, new_cat_dir, aug_idx)
                    aug_idx += 1
                    current_count = len(os.listdir(new_cat_dir))

            print(f"  [{split}/{category}] : {len(os.listdir(new_cat_dir))} gambar")

    print("\nBalanced dataset siap!")
else:
    print(f"Dataset balanced sudah ada di: {BALANCED_DATASET}")

# Ringkasan dan verifikasi folder
print("\n=== Ringkasan Dataset ===")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(BALANCED_DATASET, split)
    if not os.path.isdir(split_path):
        continue
    print(f"\n[{split.upper()}]")
    for cat in sorted(os.listdir(split_path)):
        cat_path = os.path.join(split_path, cat)
        if os.path.isdir(cat_path):
            print(f"  {cat}: {len(os.listdir(cat_path))} gambar")

# Verifikasi path sebelum training
assert os.path.isdir(os.path.join(BALANCED_DATASET, 'train')), "Folder 'train' tidak ditemukan!"
assert os.path.isdir(os.path.join(BALANCED_DATASET, 'val')), "Folder 'val' tidak ditemukan!"
print("\nVerifikasi folder OK — siap training.")


## 4. Inisialisasi Model

Menggunakan **YOLOv8s-cls** (Small) — kapasitas lebih baik dari Nano untuk 3+ kelas.


In [ ]:
model = YOLO('yolov8s-cls.pt')
print("Model YOLOv8s-cls berhasil dimuat.")


## 5. Training Model YOLOv8 (Konfigurasi Optimal)

> **Catatan**: Untuk task **klasifikasi**, YOLO mengharuskan `data` berupa **path folder** yang berisi subfolder `train/` dan `val/`. File yaml tidak didukung untuk task ini.

| Parameter | Nilai | Fungsi |
|-----------|-------|--------|
| `optimizer` | `AdamW` | Weight decay terpisah dari gradient |
| `weight_decay` | `0.0005` | L2 Regularisasi |
| `dropout` | `0.3` | Dropout di classifier head |
| `label_smoothing` | `0.1` | Cegah model terlalu overconfident |
| `warmup_epochs` | `3` | Stabilkan training di awal |
| `cos_lr` | `True` | LR turun halus mengikuti cosine curve |
| `patience` | `20` | Early stop jika tidak ada peningkatan |
| `mixup` | `0.1` | Regularisasi distribusi input |
| `erasing` | `0.3` | Cegah model fokus pada satu area |
| `amp` | `True` | Mixed precision — lebih cepat & hemat VRAM |


In [ ]:
# YOLO classification: data harus berupa PATH FOLDER, bukan file yaml
results = model.train(
    # ---- Dataset ----
    data=BALANCED_DATASET,  # Path folder (wajib untuk task classification)

    # ---- Konfigurasi Dasar ----
    epochs=100,            # Epoch banyak, early stopping yang kelola
    imgsz=224,
    batch=32,
    device=0,
    workers=4,

    # ---- Optimizer & LR ----
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    # ---- LR Scheduler ----
    cos_lr=True,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # ---- Regularisasi ----
    label_smoothing=0.1,
    dropout=0.3,

    # ---- Augmentasi Online ----
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    degrees=30.0,
    translate=0.1,
    scale=0.5,
    shear=5.0,
    flipud=0.1,
    fliplr=0.5,
    mixup=0.1,
    erasing=0.3,

    # ---- Kontrol Training ----
    patience=20,
    save_period=10,
    val=True,
    cache=False,
    amp=True,

    # ---- Output ----
    project='/kaggle/working/runs/classify',
    name='pisang_optimized',
    exist_ok=True,
)

print("\nTraining selesai!")
print(f"Top-1 Accuracy terbaik: {results.results_dict.get('metrics/accuracy_top1', 'N/A')}")


## 6. Evaluasi Model Terbaik


In [ ]:
best_pt = '/kaggle/working/runs/classify/pisang_optimized/weights/best.pt'
best_model = YOLO(best_pt)

# YOLO classification val juga butuh path folder
metrics = best_model.val(data=BALANCED_DATASET, split='val')

print("\n========================================")
print("   HASIL EVALUASI MODEL TERBAIK")
print("========================================")
print(f"  Top-1 Accuracy : {metrics.top1 * 100:.2f}%")
print(f"  Top-5 Accuracy : {metrics.top5 * 100:.2f}%")
print("========================================")


## 7. Visualisasi Kurva Training (HD)


In [ ]:
csv_path = '/kaggle/working/runs/classify/pisang_optimized/results.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    sns.set_theme(style="whitegrid", font_scale=1.2)
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    fig.suptitle('Evaluasi Training Model Klasifikasi Pisang (YOLOv8s)',
                 fontsize=18, fontweight='bold', y=1.02)

    # Plot 1: Akurasi
    axes[0].plot(df['epoch'], df['metrics/accuracy_top1'],
                 color='#2ecc71', linewidth=2.5, label='Top-1 Accuracy')
    axes[0].set_title('Akurasi per Epoch', fontsize=15, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=13)
    axes[0].set_ylabel('Akurasi', fontsize=13)
    axes[0].legend(fontsize=12)
    axes[0].set_ylim([0.8, 1.01])

    # Plot 2: Training vs Validation Loss
    axes[1].plot(df['epoch'], df['train/loss'],
                 color='#3498db', linewidth=2.5, label='Training Loss')
    axes[1].plot(df['epoch'], df['val/loss'],
                 color='#e74c3c', linewidth=2.5, linestyle='--', label='Validation Loss')
    axes[1].set_title('Training vs Validation Loss', fontsize=15, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=13)
    axes[1].set_ylabel('Loss', fontsize=13)
    axes[1].legend(fontsize=12)

    # Plot 3: Overfitting Gap
    gap = df['val/loss'] - df['train/loss']
    axes[2].fill_between(df['epoch'], gap, alpha=0.4, color='#9b59b6')
    axes[2].plot(df['epoch'], gap, color='#9b59b6', linewidth=2, label='Gap (Val - Train)')
    axes[2].axhline(y=0, color='gray', linestyle='--', linewidth=1)
    axes[2].set_title('Overfitting Gap', fontsize=15, fontweight='bold')
    axes[2].set_xlabel('Epoch', fontsize=13)
    axes[2].set_ylabel('Val Loss - Train Loss', fontsize=13)
    axes[2].legend(fontsize=12)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafik disimpan ke: /kaggle/working/training_curves.png")
else:
    print(f"File {csv_path} belum ada. Selesaikan training terlebih dahulu.")


## 8. Confusion Matrix


In [ ]:
cm_path = '/kaggle/working/runs/classify/pisang_optimized/confusion_matrix.png'
if os.path.exists(cm_path):
    plt.figure(figsize=(10, 10))
    img = mpimg.imread(cm_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Confusion Matrix Klasifikasi Pisang", fontsize=18, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
else:
    print(f"Confusion matrix belum tersedia di: {cm_path}")


## 9. Prediksi Gambar Contoh


In [ ]:
test_images = glob.glob(f"{BALANCED_DATASET}/val/*/*.jpg") + \
              glob.glob(f"{BALANCED_DATASET}/val/*/*.png")

if len(test_images) > 0:
    samples = random.sample(test_images, min(6, len(test_images)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Contoh Prediksi Model', fontsize=16, fontweight='bold')

    for ax, img_path in zip(axes.flatten(), samples):
        prediction = best_model.predict(img_path, verbose=False)
        class_idx = prediction[0].probs.top1
        class_name = prediction[0].names[class_idx]
        confidence = prediction[0].probs.top1conf.item()
        true_label = os.path.basename(os.path.dirname(img_path))

        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        color = '#2ecc71' if class_name == true_label else '#e74c3c'
        ax.set_title(
            f"Prediksi: {class_name}\n({confidence*100:.1f}%) | Label: {true_label}",
            fontsize=10, color=color, fontweight='bold'
        )
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/sample_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("Tidak ada gambar val ditemukan.")


## 10. Ekspor Model Terbaik
Menyalin `best.pt` ke root working directory agar mudah di-download dari panel Output Kaggle.


In [ ]:
import shutil

model_paths = glob.glob('/kaggle/working/runs/classify/pisang_optimized/weights/best.pt')

if model_paths:
    export_path = '/kaggle/working/best_model_pisang.pt'
    shutil.copy(model_paths[0], export_path)
    size_mb = os.path.getsize(export_path) / (1024 * 1024)

    print(f"Model disalin ke    : {export_path}")
    print(f"Ukuran model        : {size_mb:.2f} MB")
    print("\n--- INSTRUKSI WEB APP ---")
    print("1. Download 'best_model_pisang.pt' dari panel Output Kaggle.")
    print("2. Letakkan di folder 'model/' project lokal.")
    print("3. Edit app.py: MODEL_PATH = './model/best_model_pisang.pt'")
else:
    print("best.pt belum ditemukan. Pastikan training selesai!")
